In [61]:
import pandas as pd

df_customers = pd.read_csv('urbanstyle_customers.csv')
df_sales = pd.read_csv('urbanstyle_sales.csv')


In [62]:
 '''ROLL A - DATA LOADING'''

'ROLL A - DATA LOADING'

In [63]:
#liidame sales ja customers tabelid kokku, et saada juurde kliendi email ja eesnimi.
df = pd.merge(df_sales, df_customers[['customer_id', 'email', 'first_name']], on='customer_id', how='left')

In [64]:
# Laadime sales tabeli DataFrame'i ja kontrollime selle kuju (shape) ning esimeseid ridu (head)
print("Sales shape:", df_sales.shape)
display(df_sales.head())

Sales shape: (15234, 11)


,sale_id,invoice_id,sale_date,customer_id,product_id,quantity,unit_price,total_price,channel,store_location,payment_method
0,1,INV-202301-00001,2023-01-10,2588.0,1274,2,234.79,469.58,pood,Tallinn,kaart
1,2,INV-202301-00002,2023-01-16,4338.0,1207,2,241.13,482.26,pood,Pärnu,järelmaks
2,3,INV-202301-00003,2023-01-05,4673.0,1264,1,258.46,221.19,pood,Pärnu,järelmaks
3,4,INV-202301-00004,2023-01-02,4677.0,1341,3,45.21,135.63,pood,Tartu,sularaha
4,5,INV-202301-00005,2023-01-13,2390.0,1284,1,99.57,99.57,pood,Tartu,kaart


In [65]:
# Laadime customers tabeli DataFrame'i ja kontrollime selle kuju (shape) ning esimeseid ridu (head)
print("Customers shape:", df_customers.shape)
display(df_customers.head())

Customers shape: (3150, 9)


,customer_id,first_name,last_name,email,phone,city,registration_date,loyalty_tier,birth_year
0,2001,Eha,Aas,eha.aas@telia.ee,+372 8713 1455,Tallinn,2024-02-27,NaN,1973
1,2002,Aivar,Kõiv,aivar.koiv@outlook.com,+372 8943 8684,Haapsalu,2025-01-09,bronze,1988
2,2003,Maris,Rebane,maris.rebane@telia.ee,+372 5918 5726,Tartu,2021-02-03,NaN,1999
3,2004,Jaak,Talvik,jaak.talvik@mail.ee,+372 8554 4232,Tallinn,2023-11-12,silver,1974
4,2005,Raivo,Koppel,raivo.koppel@yahoo.com,+372 5298 4365,Tallinn,2023-05-22,bronze,2004


In [66]:
# Tulemuste kontroll
print("Uus shape:", df.shape)
display(df.head())

Uus shape: (15234, 13)


,sale_id,invoice_id,sale_date,customer_id,product_id,quantity,unit_price,total_price,channel,store_location,payment_method,email,first_name
0,1,INV-202301-00001,2023-01-10,2588.0,1274,2,234.79,469.58,pood,Tallinn,kaart,NaN,Hille
1,2,INV-202301-00002,2023-01-16,4338.0,1207,2,241.13,482.26,pood,Pärnu,järelmaks,merle.luik@mail.ee,Merle
2,3,INV-202301-00003,2023-01-05,4673.0,1264,1,258.46,221.19,pood,Pärnu,järelmaks,liina.saar@gmail.com,Liina
3,4,INV-202301-00004,2023-01-02,4677.0,1341,3,45.21,135.63,pood,Tartu,sularaha,aili.pihl@yahoo.com,Aili
4,5,INV-202301-00005,2023-01-13,2390.0,1284,1,99.57,99.57,pood,Tartu,kaart,triin.lill@telia.ee,Triin


In [67]:
 '''ROLL B - DATA CLEANING'''

'ROLL B - DATA CLEANING'

In [68]:
# vaatan mitu rida oli enne duplikaatite eemaldamist, vastuseks tuleb 15234 rida, mis on õige
print("Esialgne shape:", df.shape)

Esialgne shape: (15234, 13)


In [78]:
# Duplikaatide kontroll ja eemaldamine invoice_id järgi
print("Duplikaadid:", df.duplicated(subset=['invoice_id']).sum())
df = df.drop_duplicates(subset=['invoice_id'], keep='first')

Duplikaadid: 0


In [79]:
#Kontrolli NULL väärtusi:
print("NULL-id:\n", df.isnull().sum())
# Tulemus: Null väärtused on customer_id, store_location, email ja first_name veergudes. 


NULL-id:
 sale_id              0
invoice_id           0
sale_date            0
customer_id          0
product_id           0
quantity             0
unit_price           0
total_price          0
channel              0
store_location    3054
payment_method       0
email              939
first_name           0
dtype: int64


In [80]:
#Eemaldame need read, kus väärtused on null, kuna see on oluline veerg RFM analüüsi jaoks.
df = df.dropna(subset=['customer_id', 'sale_date', 'total_price'])

In [81]:
#Kuna kuupäeva formaat on erinev, siis selleks, et saaksime jooksutada järgmist koodi on alguses vaja saada formaat õigeks kus päev on esimesena märgitud, näiteks 25/01/2023, Pandas üritab automaatselt kuvada teises vormis
df['sale_date'] = pd.to_datetime(df['sale_date'], format='mixed')

In [82]:
# Konverteerime (parsime) tekstiformaadis (string) kuupäevad pandas'e jaoks õigesse kuupäeva- ja kellaajavormingusse (datetime).  
# Tänu sellele saab Pandas hiljem õigesti aru kalendrilistest kuupäevadest, mis on hädavajalik näiteks RFM-analüüsis Recency (päevade arvu viimasest ostust) arvutamiseks
df['sale_date'] = pd.to_datetime(df['sale_date'])

In [83]:
# Kontrolli outlier'id: kas on negatiivseid total_price väärtusi?
print("Negatiivsed/null väärtused:", (df['total_price'] <= 0).sum())

Negatiivsed/null väärtused: 0


In [85]:
# Eemaldame negatiivsed total_price väärtused, kuna need on ilmselt andmete sisestamise vead ja ei ole mõistlikud RFM analüüsi jaoks.
df = df[df['total_price'] > 0]

In [86]:
# Kontrollime uuesti Otlier'id: kas on negatiivseid total_price väärtusi?
print("Negatiivsed/null väärtused:", (df['total_price'] <= 0).sum())
# Tulemus on 0, mis on õige.

Negatiivsed/null väärtused: 0


In [87]:
print("--- PUHASTUSRAPORT ---")
print("Lõplik shape:", df.shape)        #Pärast duplikaatide, puuduvate väärtuste ja outlier'ite eemaldamist jäi andmestikku alles 13 468 rida ja 13 veergu
print("Unikaalsete klientide arv:", df['customer_id'].nunique())
print("Kuupäevavahemik:", df['sale_date'].min(), "kuni", df['sale_date'].max())
# Tulemus 
# 1. Müügitabeli RAW mahuks oli 15 234 rida (Roll A).
# 2. Pärast müügiarvete deduplitseerimist (invoice_id järgi) jäi alles 10 118 reaalset tehingut.
# 3. Kliendiandmete liitmisel (left join) ja subsequent NULL-väärtuste (puuduv customer_id) 
#    ning negatiivsete hindade eemaldamisel jäi RFM-analüüsiks alles täpselt 13 468 puhast koondrida.

--- PUHASTUSRAPORT ---
Lõplik shape: (8950, 13)
Unikaalsete klientide arv: 2540
Kuupäevavahemik: 2023-01-01 00:00:00 kuni 2026-09-06 00:00:00
